# Can you find Wally? 🔍

In the last workshop we looked at how a convolutional neural network (CNN) can tell us **what** is in an image. Today we go one step further and ask **where** it is.

We will build a CNN from scratch in PyTorch that looks at a busy picture and draws a box around Wally.

<br>

**What we are doing today:**

- **Making our own data** 🎨: we will paste Wally onto hundreds of different backgrounds. Because we place him ourselves, we get the answer (his box) for free.
- **Building a CNN from scratch** 🧠: convolution, pooling and a small "head" that outputs four numbers, the corners of Wally's box.
- **Writing the training loop ourselves** 🔁: forward pass, loss, `zero_grad`, `backward`, `step`. No magic `fit()` function.
- **Measuring how good our boxes are** 📏: Intersection over Union (IoU).
- **Testing on images the model has never seen** 👀: green box = where Wally really is, red box = where the model thinks he is.

<hr>

# The structure of this notebook 📖

Work through the sections **in order**. Each code cell does one small job, and the text above it explains why.

Look out for:

- **Think** 🤔: questions to discuss with the people around you. There's no code to write, but you should be able to answer them by the end.
- **Try it** 🧪: small changes you can make to see what happens.

<br>

**Tech stack used**

| Job | Library | What we use it for |
|---|---|---|
| Model, loss, training loop | [PyTorch](https://pytorch.org/) (`torch`, `torch.nn`) | We write the forward pass, `zero_grad`, `backward` and `step` ourselves |
| Batching and splitting | `torch.utils.data` | `Dataset`, `DataLoader`, `random_split` |
| Making the synthetic images | [Pillow](https://pillow.readthedocs.io/) and [NumPy](https://numpy.org/) | Pasting Wally using his transparency (alpha) mask |
| Plotting boxes and curves | [Matplotlib](https://matplotlib.org/) | `patches.Rectangle` for the true and predicted boxes |
| IoU metric | [torchvision](https://pytorch.org/vision/) | `torchvision.ops.box_iou`, which takes boxes as `(x1, y1, x2, y2)` |

<br>

**Speed tip ⚡:** In Kaggle, open the right-hand panel, go to **Settings → Accelerator** and pick a **GPU**. Training will be much faster.

<hr>

# Step 0: Setting up 🧰

First we import everything we need. Each library has one job (see the table above).

In [ ]:
import os
import csv
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.ops import box_iou

### Where our files live, and the numbers we might want to change

Putting all the settings in one place means you never have to hunt through the notebook to change something.

- The **backgrounds** are 286 crowd scenes, all 250x250 pixels.
- The **Waldos** are 5 cut-out pictures of Wally with a transparent background (PNG with an *alpha* channel).
- Our generated images will be saved in `/kaggle/working`, the folder Kaggle lets us write to.

In [ ]:
# input data
BACKGROUND_DIR = '/kaggle/input/datasets/zekikam/waldo-dataset/backgrounds'
WALDO_DIR = '/kaggle/input/datasets/zekikam/waldo-dataset/waldos'

# where we will save the images we generate
OUTPUT_DIR = '/kaggle/working/generated images'
TRAIN_DIR = os.path.join(OUTPUT_DIR, 'train images')
TEST_DIR = os.path.join(OUTPUT_DIR, 'test images')

# image size (all backgrounds are 250x250)
IMG_SIZE = 250

# how much data to make
NUM_IMAGES = 5000      # in total
NUM_TRAIN = 4000       # 80% for training
NUM_TEST = 1000        # 20% for testing

# training settings
TRAIN_IMAGES_PER_EPOCH = 1000   # the model sees the same 1000 training images every epoch
NUM_EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 5e-4

### Making our results repeatable 🎲

We use a lot of randomness today: random backgrounds, random positions, random starting weights. Setting a **seed** means everyone running this notebook gets the same random numbers, so results can be compared.

We also pick the **device**: the GPU (`cuda`) if there is one, otherwise the CPU.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

**Think** 🤔

- *If you change the seed to a different number, should the final results be exactly the same? Roughly the same?*

<hr>

# Step 1: Making our own training data 🎨

To train a model to find Wally, we need lots of pictures **and** the answer for each one: where exactly is Wally? Labelling thousands of real pictures by hand would take days.

Instead we will **make** our pictures. We take a background, paste Wally somewhere on it, and write down where we put him. The labels come for free! This is called **synthetic data**.

<br>

### 1.1 Loading the backgrounds

Let's find all the background files and look at a few.

In [ ]:
background_paths = sorted(
    os.path.join(BACKGROUND_DIR, f) for f in os.listdir(BACKGROUND_DIR)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
)
print('Number of backgrounds:', len(background_paths))

# load them all into memory once, so we don't keep re-reading files from disk
backgrounds = [Image.open(p).convert('RGB') for p in background_paths]
print('Size of the first background:', backgrounds[0].size)

In [ ]:
# show 5 random backgrounds
fig, axes = plt.subplots(1, 5, figsize = (20, 4))
for ax, bg in zip(axes, random.sample(backgrounds, 5)):
    ax.imshow(bg)
    ax.axis('off')
plt.suptitle('5 random backgrounds')
plt.show()

**Think** 🤔

- *Why do we want lots of **different** backgrounds rather than one? What could go wrong if every training picture had the same background?*

### 1.2 Loading the Waldos

Each Waldo is a PNG with 4 channels: **R**ed, **G**reen, **B**lue and **A**lpha. The alpha channel says how see-through each pixel is: 0 means fully transparent, 255 means fully solid. That's what lets us paste Wally without a white rectangle around him.

In [ ]:
waldo_paths = sorted(
    os.path.join(WALDO_DIR, f) for f in os.listdir(WALDO_DIR)
    if f.lower().endswith('.png')
)

# convert('RGBA') makes sure every Waldo has an alpha channel
waldos = [Image.open(p).convert('RGBA') for p in waldo_paths]

for p, w in zip(waldo_paths, waldos):
    print(os.path.basename(p), 'size (width, height):', w.size)

In [ ]:
# show all the Waldos
fig, axes = plt.subplots(1, len(waldos), figsize = (3 * len(waldos), 4))
for i, (ax, w) in enumerate(zip(axes, waldos)):
    ax.imshow(w)
    ax.set_title(f'Waldo {i}: {w.size[0]}x{w.size[1]}')
    ax.axis('off')
plt.show()

### 1.3 Changing Wally a little each time 🎭

If we pasted exactly the same 5 Waldos every time, the model might just memorise those exact pixels. Real Wallys are drawn a bit differently each time, so we make every pasted Wally slightly different:

- a small **rotation** (up to ±10°)
- a **brightness** change (a bit darker or lighter)
- a **colour shift** (a little more or less red, green or blue)

This is called **data augmentation**.

The colour shift uses NumPy: we turn the image into an array of numbers with shape `(height, width, 4)` and multiply the R, G and B channels by slightly different amounts. We leave the alpha channel alone.

In [ ]:
def augment_waldo(waldo):
    # 1. small random rotation. expand=True grows the canvas so his feet and hat aren't cut off
    angle = random.uniform(-10, 10)
    waldo = waldo.rotate(angle, expand = True, resample = Image.BICUBIC)

    # 2. random brightness: 1.0 = unchanged, 0.8 = darker, 1.2 = brighter
    rgb = waldo.convert('RGB')
    alpha = waldo.getchannel('A')
    rgb = ImageEnhance.Brightness(rgb).enhance(random.uniform(0.8, 1.2))

    # 3. random colour shift: multiply each of R, G, B by its own number near 1
    pixels = np.array(rgb).astype(np.float32)
    shift = np.random.uniform(0.9, 1.1, size = 3)
    pixels = np.clip(pixels * shift, 0, 255).astype(np.uint8)

    # put the alpha (transparency) channel back on
    waldo = Image.fromarray(pixels)
    waldo.putalpha(alpha)
    return waldo

In [ ]:
# see what augmentation does: the same Waldo, 6 times
fig, axes = plt.subplots(1, 6, figsize = (15, 4))
for ax in axes:
    ax.imshow(augment_waldo(waldos[0]))
    ax.axis('off')
plt.suptitle('One Waldo, augmented 6 different ways')
plt.show()

**Think** 🤔

- *Why do we only rotate by a **small** amount? What would happen to the box if we rotated Wally by 90°?*
- *Can you think of an augmentation that would be a bad idea for finding Wally? (Hint: what if we turned his red stripes green?)*

### 1.4 Pasting Wally onto a background 📌

Now we put it together. For one sample image we:

1. pick a random background
2. pick a random Waldo and augment it
3. pick a random top-left corner `(x, y)` so that Wally fits fully inside the picture
4. paste him using his alpha channel as the **mask** (only the solid pixels are copied)
5. work out his **bounding box**

<br>

**Bounding boxes** 📦

We describe a box with its top-left and bottom-right corners: `(x1, y1, x2, y2)`. This is the format `torchvision.ops.box_iou` wants.

In images, `x` counts pixels from the left and `y` counts pixels **down from the top**.

Because we rotated Wally, his picture now has transparent corners. `getbbox()` on the alpha channel gives the smallest box around the pixels that are actually Wally, so our box fits him tightly.

In [ ]:
def make_sample():
    # 1. random background (copy it, so we don't draw on the original)
    background = random.choice(backgrounds).copy()

    # 2. random Waldo, made slightly different
    waldo = augment_waldo(random.choice(waldos))
    w, h = waldo.size

    # 3. random position where Waldo fits completely inside the image
    x = random.randint(0, IMG_SIZE - w)
    y = random.randint(0, IMG_SIZE - h)

    # 4. paste: the 'mask' argument uses Waldo's alpha channel
    background.paste(waldo, (x, y), mask = waldo)

    # 5. bounding box of the visible part of Waldo, moved to where we pasted him
    left, top, right, bottom = waldo.getchannel('A').getbbox()
    box = (x + left, y + top, x + right, y + bottom)

    return background, box

### 1.5 Drawing boxes 🟩🟥

We will use **green** for the true box (where we put Wally) and later **red** for the box the model predicts.

Matplotlib's `patches.Rectangle` wants the top-left corner, then the **width** and **height**, so we convert from `(x1, y1, x2, y2)`.

In [ ]:
def draw_box(ax, box, colour):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                             linewidth = 2, edgecolor = colour, facecolor = 'none')
    ax.add_patch(rect)

In [ ]:
# 5 random generated images with a green box around Waldo
fig, axes = plt.subplots(1, 5, figsize = (20, 4))
for ax in axes:
    image, box = make_sample()
    ax.imshow(image)
    draw_box(ax, box, 'green')
    ax.set_title(f'box = {box}')
    ax.axis('off')
plt.show()

**Think** 🤔

- *Can **you** find Wally without the green box? How long does it take you?*
- *Why does `x` only go up to `IMG_SIZE - w` and not `IMG_SIZE`?*

### 1.6 Making 5000 images and splitting them 80/20 ✂️

We make all 5000 images up front and save them as files, the same as a real dataset.

Then we split them:

- **Training set (80%, 4000 images):** the model learns from these.
- **Test set (20%, 1000 images):** we keep these hidden until the end, to check the model works on pictures it has **never seen**.

`random_split` shuffles the numbers 0 to 4999 and deals them into two groups of 4000 and 1000. We give it its own seeded `generator` so the split is the same every time.

Each folder also gets a `labels.csv` file listing every image and its box.

In [ ]:
os.makedirs(TRAIN_DIR, exist_ok = True)
os.makedirs(TEST_DIR, exist_ok = True)

# decide which image numbers go to training and which go to testing
train_ids, test_ids = random_split(range(NUM_IMAGES), [NUM_TRAIN, NUM_TEST],
                                   generator = torch.Generator().manual_seed(SEED))
test_ids = set(test_ids)
print('Training images:', len(train_ids), ' Test images:', len(test_ids))

In [ ]:
tick = time.time()
train_rows, test_rows = [], []

for i in range(NUM_IMAGES):
    image, box = make_sample()
    filename = f'image_{i:04d}.png'

    if i in test_ids:
        image.save(os.path.join(TEST_DIR, filename))
        test_rows.append([filename, *box])
    else:
        image.save(os.path.join(TRAIN_DIR, filename))
        train_rows.append([filename, *box])

# save the labels next to the images
for folder, rows in [(TRAIN_DIR, train_rows), (TEST_DIR, test_rows)]:
    with open(os.path.join(folder, 'labels.csv'), 'w', newline = '') as f:
        writer = csv.writer(f)
        writer.writerow(['filename', 'x1', 'y1', 'x2', 'y2'])
        writer.writerows(rows)

print(f'Made {NUM_IMAGES} images in {time.time() - tick:.1f} seconds')
print('Train folder:', len(os.listdir(TRAIN_DIR)) - 1, 'images')
print('Test folder: ', len(os.listdir(TEST_DIR)) - 1, 'images')

**Think** 🤔

- *Why must the test images never be used for training?*
- *Every image uses one of the same 286 backgrounds. Is it possible for a background to appear in both the training and the test set? Does that matter?*

<hr>

# Step 2: Feeding images to PyTorch 🍽️

A neural network only understands numbers, and PyTorch wants them in **tensors**. We need two helpers from `torch.utils.data`:

- A **`Dataset`** answers two questions: *how many examples are there?* (`__len__`) and *give me example number i* (`__getitem__`).
- A **`DataLoader`** takes a `Dataset` and serves it in shuffled **batches**.

<br>

### 2.1 Our Wally `Dataset`

For example number `i`, our dataset:

1. opens the image file
2. turns it into a tensor of shape `(3, 250, 250)`, **channels first**, which is what PyTorch convolution layers expect (NumPy and PIL put channels last)
3. scales pixel values from 0–255 down to 0–1
4. scales the box from pixels down to 0–1 by dividing by the image size

<br>

**Why scale the box to 0–1?** Our model will output numbers that start out small (close to 0). If the targets were in pixels (up to 250), the model's first guesses would be off by about 100, the squared error would be about 10,000, and training would be wild and unstable. Keeping inputs and outputs on a small, similar scale makes learning smooth.

In [ ]:
class WaldoDataset(Dataset):
    def __init__(self, folder):
        self.folder = folder
        with open(os.path.join(folder, 'labels.csv')) as f:
            reader = csv.reader(f)
            next(reader)  # skip the header row
            self.rows = [(r[0], [float(v) for v in r[1:]]) for r in reader]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        filename, box = self.rows[i]
        image = Image.open(os.path.join(self.folder, filename)).convert('RGB')

        # (height, width, 3) with values 0-255  ->  (3, height, width) with values 0-1
        image = torch.tensor(np.array(image), dtype = torch.float32).permute(2, 0, 1) / 255

        # box in pixels -> box as a fraction of the image size (0-1)
        box = torch.tensor(box, dtype = torch.float32) / IMG_SIZE
        return image, box

In [ ]:
train_dataset = WaldoDataset(TRAIN_DIR)
test_dataset = WaldoDataset(TEST_DIR)

image, box = train_dataset[0]
print('Number of training images:', len(train_dataset))
print('Number of test images:    ', len(test_dataset))
print('One image tensor has shape:', image.shape)
print('Pixel values go from', image.min().item(), 'to', image.max().item())
print('Its box (0-1):', box)

### 2.2 Using the same 1000 training images every epoch

An **epoch** is one pass through the training data. Our plan is 10 epochs of 1000 images, and every epoch uses the **same** 1000 images. We pick them once with `random_split` and keep the rest aside.

*Try it* 🧪: set `TRAIN_IMAGES_PER_EPOCH = 4000` at the top to train on all 4000 images. It takes 4x longer per epoch. Does the model get better?

In [ ]:
train_subset, _ = random_split(train_dataset,
                               [TRAIN_IMAGES_PER_EPOCH, len(train_dataset) - TRAIN_IMAGES_PER_EPOCH],
                               generator = torch.Generator().manual_seed(SEED))
print('Images used in each epoch:', len(train_subset))

### 2.3 Batches with a `DataLoader`

We don't feed images in one at a time, or all 1000 at once. We use **batches** of 32:

- One at a time is slow, and each update is based on just one noisy example.
- All at once might not fit in memory, and we'd only get one update per epoch.

`shuffle=True` mixes up the order every epoch so the model doesn't learn anything from the order of the images. We don't shuffle the test set because the order doesn't matter there.

In [ ]:
train_loader = DataLoader(train_subset, batch_size = BATCH_SIZE, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)

images, boxes = next(iter(train_loader))
print('A batch of images has shape:', images.shape, ' -> (batch, channels, height, width)')
print('A batch of boxes has shape: ', boxes.shape, ' -> (batch, 4 numbers per box)')
print('Batches per epoch:', len(train_loader))

**Think** 🤔

- *With 1000 images and a batch size of 32, why are there 32 batches and not 31.25?*
- *The weights are updated once per batch. How many updates happen in 10 epochs?*

<hr>

# Step 3: Building the CNN from scratch 🧠

Our model has two parts:

1. **The feature extractor (convolution + pooling):** turns the picture into a small grid of "features" (things like *red and white stripes here*, *round glasses there*).
2. **The head (fully connected layers):** turns those features into 4 numbers, the corners of Wally's box `(x1, y1, x2, y2)`.

<br>

### The building block: Conv → ReLU → MaxPool

- **`nn.Conv2d(in, out, 3, padding=1)`** slides `out` small 3x3 filters (kernels) over the image. Each filter learns to spot one pattern. The **same** filter is used at every position, so a stripe detector works wherever Wally is. `padding=1` keeps the width and height the same.
- **`nn.ReLU()`** keeps positive values and sets negatives to 0. Without it, stacking layers would be no more powerful than a single layer.
- **`nn.MaxPool2d(2)`** keeps the largest value in each 2x2 square. This halves the width and height, so later layers see a bigger area of the picture and there's less to compute.

<br>

### How the size changes

| After | Shape (channels x height x width) |
|---|---|
| input | 3 x 250 x 250 |
| block 1 (16 filters) | 16 x 125 x 125 |
| block 2 (32 filters) | 32 x 62 x 62 |
| block 3 (64 filters) | 64 x 31 x 31 |
| block 4 (64 filters) | 64 x 15 x 15 |
| flatten | 14,400 |
| linear + ReLU | 64 |
| linear | 4 (the box) |

The flatten step keeps every position of the 15x15 grid, so the head still knows **where** each feature was found. That's exactly what we need for the box.

In [ ]:
class WaldoFinder(nn.Module):
    def __init__(self):
        super().__init__()

        # feature extractor: 4 blocks of Conv -> ReLU -> MaxPool
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),                       # 250 -> 125

            nn.Conv2d(16, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),                       # 125 -> 62

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),                       # 62 -> 31

            nn.Conv2d(64, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),                       # 31 -> 15
        )

        # head: flatten the 64 x 15 x 15 grid and predict 4 numbers
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 15 * 15, 64),
            nn.ReLU(),
            nn.Linear(64, 4),                      # (x1, y1, x2, y2), each 0-1
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x

In [ ]:
model = WaldoFinder().to(device)
print(model)

num_params = sum(p.numel() for p in model.parameters())
print(f'\nNumber of weights and biases: {num_params:,}')

### Checking the shapes ✅

Before training, it's good practice to push a fake batch through the model and check the output shape is what we expect: one box of 4 numbers for each image.

In [ ]:
fake_batch = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(device)
print('Feature grid shape:', model.features(fake_batch).shape)
print('Output shape:      ', model(fake_batch).shape)

**Think** 🤔

- *Count the weights in the first `nn.Linear` layer (14,400 inputs x 64 outputs, plus 64 biases). What fraction of the whole model is that one layer?*
- *A first conv layer has 3 x 3 x 3 x 16 weights + 16 biases = 448 parameters, yet it looks at all 62,500 pixels. How is that possible? Why couldn't a plain fully connected network do the same with so few weights?*
- *What would happen to the size of the first linear layer if we removed one of the MaxPool layers?*

<hr>

# Step 4: The loss function and the optimiser 📉

### Loss: how wrong is the model?

The **loss** turns "how bad is this guess?" into a single number. Training means making that number smaller.

We use **mean squared error (MSE)**: for every image, take the 4 predicted numbers, subtract the 4 true numbers, square the differences and average them all.

$$\text{MSE} = \frac{1}{N}\sum (\text{predicted} - \text{true})^2$$

Squaring makes every error positive and punishes big mistakes much more than small ones.

<br>

### Optimiser: how do we fix it?

The **optimiser** decides how to nudge each weight to reduce the loss. We use **Adam**, which adapts its step size for each weight separately and usually works well without much tuning.

The **learning rate** is how big those nudges are. Too big and training jumps around or gets stuck; too small and it takes forever. We use 0.0005.

In [ ]:
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)

**Think** 🤔

- *Our boxes are scaled 0–1. If the model is off by 0.1 on every coordinate, what is the MSE? How many pixels is that?*
- *Why might a learning rate that's too large make training get "stuck"?*

<hr>

# Step 5: Training the model 🔁

This is the heart of deep learning, and in PyTorch we write it ourselves. For every **batch**:

1. **Forward pass:** `predictions = model(images)`. The images go through the network and out come predicted boxes.
2. **Loss:** `loss = loss_function(predictions, boxes)`. How wrong were we?
3. **`optimizer.zero_grad()`:** clear the gradients from the previous batch. PyTorch **adds** new gradients onto old ones, so if we forget this, old batches keep affecting new updates.
4. **`loss.backward()`:** backpropagation. PyTorch works out, for every weight, which direction would make the loss smaller (the gradient).
5. **`optimizer.step()`:** nudge every weight a little in that direction.

So the weights are updated **once per batch**: 32 times per epoch, 320 times in total.

<br>

After each epoch we also check the loss on the **test** set, without updating any weights. If the training loss keeps falling but the test loss starts rising, the model is memorising the training images rather than learning to find Wally (**overfitting**).

- `model.train()` and `model.eval()` switch the model between training and testing mode. Our model has no layers that behave differently (like Dropout), but it's a good habit.
- `torch.no_grad()` tells PyTorch not to track gradients, which saves memory and time when we're only testing.

In [ ]:
def test_loss():
    model.eval()
    total = 0
    with torch.no_grad():
        for images, boxes in test_loader:
            images, boxes = images.to(device), boxes.to(device)
            total += loss_function(model(images), boxes).item() * len(images)
    return total / len(test_dataset)

In [ ]:
train_losses, test_losses = [], []
tick = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0

    for images, boxes in train_loader:
        images, boxes = images.to(device), boxes.to(device)

        predictions = model(images)                  # 1. forward pass
        loss = loss_function(predictions, boxes)     # 2. how wrong are we?

        optimizer.zero_grad()                        # 3. clear old gradients
        loss.backward()                              # 4. backpropagation
        optimizer.step()                             # 5. update the weights

        running_loss += loss.item() * len(images)

    train_losses.append(running_loss / len(train_subset))
    test_losses.append(test_loss())
    print(f'Epoch {epoch + 1:2d}/{NUM_EPOCHS}  train loss: {train_losses[-1]:.4f}  '
          f'test loss: {test_losses[-1]:.4f}  ({time.time() - tick:.0f} s so far)')

print(f'\nTraining took {(time.time() - tick) / 60:.1f} minutes')

### Plotting the learning curves 📈

A **learning curve** shows the loss after each epoch. We want both lines to go down. A big gap where the test loss is much higher than the training loss is a sign of overfitting.

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
plt.figure(figsize = (8, 5))
plt.plot(epochs, train_losses, 'o-', label = 'training loss')
plt.plot(epochs, test_losses, 'o-', label = 'test loss')
plt.xlabel('epoch')
plt.ylabel('mean squared error')
plt.title('Learning curves')
plt.legend()
plt.show()

**Think** 🤔

- *What would happen if we deleted `optimizer.zero_grad()`? Try it!* 🧪
- *Is there a gap between the training and test loss? What does that tell you?*
- *The loss is a tiny number like 0.002. Is that good? What does it mean in pixels? (Hint: take the square root, then multiply by 250.)*

<hr>

# Step 6: How good are the boxes? IoU 📏

The loss tells us the model is improving, but "MSE = 0.002" is hard to picture. A better question is: **how much do the predicted and true boxes overlap?**

<br>

### Intersection over Union (IoU)

$$\text{IoU} = \frac{\text{area where the two boxes overlap}}{\text{total area covered by the two boxes together}}$$

- **1** means a perfect match.
- **0** means the boxes don't touch at all.
- A common rule of thumb: **IoU > 0.5 counts as a correct detection**.

<br>

`torchvision.ops.box_iou(boxes1, boxes2)` takes boxes as `(x1, y1, x2, y2)` and returns a table comparing **every** box in the first list with **every** box in the second. We only want each prediction compared with its **own** true box, so we take the **diagonal** of that table.

In [ ]:
# a quick example with two boxes we can work out by hand
true_box = torch.tensor([[0., 0., 10., 10.]])   # a 10x10 square
pred_box = torch.tensor([[5., 0., 15., 10.]])   # the same square moved 5 pixels right
print('IoU =', box_iou(true_box, pred_box).item())

**Think** 🤔

- *Work out the IoU above by hand. Overlap area = ? Union area = ?*
- *Wally is about 25x50 pixels. Roughly how many pixels can a prediction be off sideways before the IoU drops below 0.5?*

In [ ]:
model.eval()
all_predictions, all_boxes = [], []
with torch.no_grad():
    for images, boxes in test_loader:
        all_predictions.append(model(images.to(device)).cpu())
        all_boxes.append(boxes)

# back to pixels
pred_boxes = torch.cat(all_predictions) * IMG_SIZE
true_boxes = torch.cat(all_boxes) * IMG_SIZE

ious = box_iou(pred_boxes, true_boxes).diagonal()
print(f'Mean IoU on the {len(ious)} test images: {ious.mean():.2f}')
print(f'Correct detections (IoU > 0.5): {(ious > 0.5).float().mean():.1%}')

In [ ]:
plt.figure(figsize = (8, 5))
plt.hist(ious.numpy(), bins = 20, range = (0, 1))
plt.axvline(0.5, color = 'red', linestyle = '--', label = 'IoU = 0.5')
plt.xlabel('IoU')
plt.ylabel('number of test images')
plt.title('How well do the predicted boxes overlap the true boxes?')
plt.legend()
plt.show()

<hr>

# Step 7: Testing on unseen images 👀

The moment of truth! These are test images the model has **never** trained on.

- 🟩 **Green box:** where Wally really is
- 🟥 **Red box:** where the model thinks he is

Under each picture we show the IoU and the **error**: how many pixels the centre of the red box is from the centre of the green box.

Run the cell a few times to see different images.

In [ ]:
def box_centre(box):
    x1, y1, x2, y2 = box
    return (x1 + x2) / 2, (y1 + y2) / 2

fig, axes = plt.subplots(2, 4, figsize = (20, 10))
for ax, i in zip(axes.flatten(), random.sample(range(len(test_dataset)), 8)):
    image, _ = test_dataset[i]
    true_box, pred_box = true_boxes[i].tolist(), pred_boxes[i].tolist()

    (tx, ty), (px, py) = box_centre(true_box), box_centre(pred_box)
    error = ((tx - px) ** 2 + (ty - py) ** 2) ** 0.5

    ax.imshow(image.permute(1, 2, 0))       # back to (height, width, channels) for plotting
    draw_box(ax, true_box, 'green')
    draw_box(ax, pred_box, 'red')
    ax.set_title(f'IoU = {ious[i]:.2f}   error = {error:.1f} px')
    ax.axis('off')
plt.show()

### The hardest test images 😬

It's often more useful to look at where the model **fails**. These are the 4 test images with the lowest IoU.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize = (20, 5))
for ax, i in zip(axes, ious.argsort()[:4].tolist()):
    image, _ = test_dataset[i]
    ax.imshow(image.permute(1, 2, 0))
    draw_box(ax, true_boxes[i].tolist(), 'green')
    draw_box(ax, pred_boxes[i].tolist(), 'red')
    ax.set_title(f'IoU = {ious[i]:.2f}')
    ax.axis('off')
plt.show()

**Think** 🤔

- *Look at the worst images. Why do you think the model got these wrong? Is it the background, the Waldo, or where he was placed?*
- *Our model always outputs exactly one box. What would it do with a picture that has **no** Wally in it? With **two** Wallys?*
- *We trained only on pasted Wallys. Do you expect it to work on a real page from a Where's Wally book? Why or why not?*

<hr>

# Where next? 🚀

Some things to try if you finish early:

- 🧪 Train on all 4000 images (`TRAIN_IMAGES_PER_EPOCH = 4000`) or for more epochs. Does the IoU go up?
- 🧪 Change the learning rate to `1e-2` or `1e-5`. What happens to the learning curves?
- 🧪 Remove a MaxPool layer (you'll need to fix the `64 * 15 * 15` in the head). What happens to the number of parameters and the training time?
- 🧪 Add stronger augmentation, like bigger rotations or making Wally smaller. Does the model still find him?
- 🧠 Add a second head that answers "is Wally in this picture at all?" and train it on some images without Wally.